# tokenbench on a T4: cross-checking servellm, then an open-loop rate sweep

GPU session required (Kaggle T4). ~10 min install, ~5 min load, ~20 min runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
%pip install -q "git+https://github.com/NomanJafar/servellm"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio   # the import chain `vllm serve` needs; fail here, not later
print("vllm", vllm.__version__, "torch", torch.__version__, "tokenbench", tokenbench.__version__)

## Launch vLLM (same config as servellm)

In [ ]:
from servellm.launch import ServerConfig, VllmServer
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096), log_path="results/startup.log")
server.start(ready_timeout_s=900)

## 1. Closed-loop sweep (should reproduce servellm's numbers)

In [ ]:
!tokenbench sweep --base-url http://127.0.0.1:8000 --model {MODEL} --out results/closed.json --levels 1,2,4,8,16,32,64 --num-requests 32 --prompt-tokens 256 --max-tokens 128 --slo-ttft-ms 500 --slo-tpot-ms 50 --plots

## 2. Open-loop rate sweep with the sample prompt distribution

Finds the arrival rate at which TTFT stops being bounded. 45 s per level.

In [ ]:
!tokenbench sweep --base-url http://127.0.0.1:8000 --model {MODEL} --out results/open.json --mode open --levels 0.5,1,2,4,8,16 --duration 45 --prompts sample --max-tokens 256 --slo-ttft-ms 1000 --slo-tpot-ms 50 --plots

In [ ]:
server.stop()

## 3. Look

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/closed.md").read()))
display(Markdown(open("results/open.md").read()))
for f in ["throughput_vs_load.png", "latency_vs_load.png", "goodput_vs_load.png"]:
    display(Image("results/" + f))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip